In [1]:
import optuna
import numpy as np
import pandas as pd
import os

from sklearn.preprocessing import StandardScaler

from adbench.run_new import RunPipeline
from adbench.myutils_new import Utils
from MSML_v9 import MSML


# -------------------------------------------------
# Optuna Objective (FULL ADBench protocol)
# -------------------------------------------------
def objective(trial):

    # ---------- Hyperparameter Search Space ----------
    k = trial.suggest_int("k", 10, 100)
    nbd_sample_count_threshold = trial.suggest_int(
        "nbd_sample_count_threshold", 5, 80
    )
    learning_rate = trial.suggest_float(
        "learning_rate", 0.05, 1.0, log=True
    )
    max_iters_shift = trial.suggest_int("max_iters_shift", 5, 20)
    shift_threshold = trial.suggest_float(
        "shift_threshold", 1e-5, 1e-2, log=True
    )
    anomalyThreshold = trial.suggest_float(
        "anomalyThreshold", 0.01, 0.3
    )

    # ---------- Customized MSML Wrapper ----------
    class OptunaMSML(MSML):
        def __init__(self, seed, model_name=None):
            super().__init__(
                seed=seed,
                k=k,
                nbd_sample_count_threshold=nbd_sample_count_threshold,
                learning_rate=learning_rate,
                max_iters_shift=max_iters_shift,
                shift_threshold=shift_threshold,
                anomalyThreshold=anomalyThreshold,
                scaler=StandardScaler()
            )

    # ---------- ADBench Pipeline (FULL SETTING) ----------
    pipeline = RunPipeline(
        suffix="Optuna_MSML_FULL",
        parallel="unsupervise",
        realistic_synthetic_mode="cluster",
        noise_type=None
    )

    # ---------- Run FULL ADBench ----------
    pipeline.run(clf=OptunaMSML)

    # ---------- Load AUCROC Results ----------
    result_path = os.path.join(
    "adbench",
    "result",
    f"AUCROC_{pipeline.suffix}.csv"
    )

    df_aucroc = pd.read_csv(result_path, index_col=0)

    # ---------- Compute Mean AUCROC ----------
    mean_aucroc = np.nanmean(df_aucroc.values)

    return float(mean_aucroc)


# -------------------------------------------------
# Optuna Callback (print after each trial)
# -------------------------------------------------
def print_trial_result(study, trial):
    print("\n================ Trial Finished ================")
    print(f"Trial number : {trial.number}")
    print(f"AUCROC       : {trial.value}")
    print("Hyperparameters:")
    for k, v in trial.params.items():
        print(f"  {k}: {v}")
    print("================================================\n")


# -------------------------------------------------
# Main
# -------------------------------------------------
if __name__ == "__main__":

    utils = Utils()
    utils.download_datasets()

    study = optuna.create_study(
        direction="maximize",
        study_name="MSML_AUCROC_ADBench_FULL"
    )

    study.optimize(
        objective,
        n_trials=10,
        callbacks=[print_trial_result]
    )

    print(" Best AUCROC:", study.best_value)
    print(" Best hyperparameters:", study.best_params)

    df = study.trials_dataframe()
    df.to_csv(
        "adbench/result/MSDE_optuna_cluster_none_noise.csv",
        index=False
    )

    print(" Saved to adbench/result/MSDE_optuna_cluster_none_noise.csv")


if there is any question while downloading datasets, we suggest you to download it from the website:
https://github.com/Minqi824/ADBench/tree/main/adbench/datasets
如果您在中国大陆地区，请使用链接：
https://jihulab.com/BraudoCC/ADBench_datasets/
100% [................................................................................] 3852 / 3852

100%|███████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 450.48it/s]
[I 2026-01-08 10:42:39,274] A new study created in memory with name: MSML_AUCROC_ADBench_FULL


CIFAR10_0.npz already exists. Skipping download...
CIFAR10_1.npz already exists. Skipping download...
CIFAR10_2.npz already exists. Skipping download...
CIFAR10_3.npz already exists. Skipping download...
CIFAR10_4.npz already exists. Skipping download...
CIFAR10_5.npz already exists. Skipping download...
CIFAR10_6.npz already exists. Skipping download...
CIFAR10_7.npz already exists. Skipping download...
CIFAR10_8.npz already exists. Skipping download...
CIFAR10_9.npz already exists. Skipping download...
FashionMNIST_0.npz already exists. Skipping download...
FashionMNIST_1.npz already exists. Skipping download...
FashionMNIST_2.npz already exists. Skipping download...
FashionMNIST_3.npz already exists. Skipping download...
FashionMNIST_4.npz already exists. Skipping download...
FashionMNIST_5.npz already exists. Skipping download...
FashionMNIST_6.npz already exists. Skipping download...
FashionMNIST_7.npz already exists. Skipping download...
FashionMNIST_8.npz already exists. Skippin

0it [00:00, ?it/s]

generating duplicate samples for dataset 15_Hepatitis...


...

Current experiment parameters: ('9_census', 0.0, np.int64(1)), model: Customized, metrics: {'aucroc': np.float64(0.6084547069271758), 'aucpr': np.float64(0.07522410825142914), 'p_at_n': np.float64(0.02702702702702703), 'adj_p_at_n': np.float64(-0.03691613460707599), 'adj_ap': np.float64(0.014448427976656275)}, fitting time: 1.430511474609375e-06, inference time: 26.68341565132141
subsampling for dataset 9_census...
current noise type: None
{'Samples': 10000, 'Features': 500, 'Anomalies': np.int64(628), 'Anomalies Ratio(%)': np.float64(6.28)}
Model: Customized, AUC-ROC: 0.585049635301595, AUC-PR: 0.08283276868965098


1082it [21:30,  5.71s/it]

Current experiment parameters: ('9_census', 0.0, np.int64(2)), model: Customized, metrics: {'aucroc': np.float64(0.585049635301595), 'aucpr': np.float64(0.08283276868965098), 'p_at_n': np.float64(0.05319148936170213), 'adj_p_at_n': np.float64(-0.010108652885808535), 'adj_ap': np.float64(0.021514333594933478)}, fitting time: 1.1920928955078125e-06, inference time: 26.92098832130432
subsampling for dataset 9_census...
current noise type: None
{'Samples': 10000, 'Features': 500, 'Anomalies': np.int64(652), 'Anomalies Ratio(%)': np.float64(6.52)}
Model: Customized, AUC-ROC: 0.7181031471076306, AUC-PR: 0.17430807885762462


1104it [23:03,  1.25s/it]
[I 2026-01-08 14:25:01,543] Trial 9 finished with value: 0.9322997957224556 and parameters: {'k': 96, 'nbd_sample_count_threshold': 18, 'learning_rate': 0.3108571174988493, 'max_iters_shift': 10, 'shift_threshold': 8.189170861549638e-05, 'anomalyThreshold': 0.27022671226162037}. Best is trial 6 with value: 0.9441062841033334.


Current experiment parameters: ('9_census', 0.0, np.int64(3)), model: Customized, metrics: {'aucroc': np.float64(0.7181031471076306), 'aucpr': np.float64(0.17430807885762462), 'p_at_n': np.float64(0.20408163265306123), 'adj_p_at_n': np.float64(0.14844682523508693), 'adj_ap': np.float64(0.11659209578205203)}, fitting time: 1.6689300537109375e-06, inference time: 27.775974988937378

================ Trial Finished ================
Trial number : 9
AUCROC       : 0.9322997957224556
Hyperparameters:
  k: 96
  nbd_sample_count_threshold: 18
  learning_rate: 0.3108571174988493
  max_iters_shift: 10
  shift_threshold: 8.189170861549638e-05
  anomalyThreshold: 0.27022671226162037

 Best AUCROC: 0.9441062841033334
 Best hyperparameters: {'k': 82, 'nbd_sample_count_threshold': 67, 'learning_rate': 0.11713494302520987, 'max_iters_shift': 7, 'shift_threshold': 0.00026919592075234797, 'anomalyThreshold': 0.13763350526690374}
 Saved to adbench/result/MSDE_optuna_cluster_none_noise.csv
